# Competitor Analysis Using Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System

## Business Context
Performing competitor analysis with a Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System.

## Problem Scenario
Conducting a robust competitor analysis of a company is a time-consuming and research-intensive process that may be prone to errors for example, outdated information.

Analysts spend significant time gathering data, identifying relevant competitors, and synthesizing insights into actionable recommendations.

## Objective
To address above mentioned challenges, a Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System is proposed.

This system will perform competitor analysis of a given company by comparing it with its key rivals in the relevant industry using the latest available web data.

Users will input a company name (e.g., "Tesla") and receive a well-structured report, comparing that company to its primary competitors (e.g., in the electric vehicle sector for Tesla), by leveraging a multi-agent collaborative architecture powered by RAG capabilities.

The system should deliver rapid, accurate, and actionable insights.

## Solution Approach

A Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System is implemented using a sequential workflow pattern.

This system automates competitor analysis by breaking down the task into distinct steps handled by specialized agents.

The sequential pattern ensures each step builds on the previous one. Also, incorporating RAG to retrieve and augment data from a vector database for context-aware insights.


**Below are the key components of the implementation:**

### State Management

The workflow uses a Pydantic-based state `CompetitiveAnalysisState` to track variables like company name, generated questions, search results, vectorstore status, and the final report.

### Tools
- `suggest_questions`: Generates relevant questions for competitor analysis using the LLM.
- `fetch_search_results`: Searches the web via Tavily API to fetch answers for the questions.
- `store_in_chromadb`: Stores question-answer pairs in ChromaDB for efficient retrieval.
- `generate_report`: Uses RAG to query the vectorstore and draft a structured report.




### Agents

- `Question Generator Agent`: Validates the company, identifies its sector, and generates analysis questions.

- `Data Retrieval and Storage Agent`: Fetches answers from the web and stores them in the vector database.

- `Report Drafter Agent`: Retrieves stored data via RAG and generates a professional, actionable report with sections like Executive Summary, Company Overview, Key Competitors, Strengths/Weaknesses, Market Strategies, and Recommendations.




### Workflow (Sequential Pattern)

The system uses LangGraph's StateGraph to define nodes for each agent.
Edges connect the nodes sequentially as indicated below.

`START → Question Generation → Data Retrieval/Storage → Report Drafting → END.`

The final output is formatted competitive analysis report, displayed in Markdown for readability.

# Installing the Libraries

Installing the libraries:
- **openai==1.99.9** → Official OpenAI client library for interacting with GPT models.  
- **langchain==0.3.27** → Core framework for building applications powered by LLMs.  
- **langchain-openai==0.3.30** → LangChain integration for OpenAI models.  
- **langchain-community==0.3.27** → Community-contributed LangChain modules (tools, integrations).  
- **langgraph==0.6.4** → Build and manage multi-step workflows or agent graphs for LLM-powered systems.  
- **langchain-chroma==0.2.5** → Connector for using **ChromaDB** as a vector database with LangChain.  
- **chromadb==1.0.16** → Open-source vector database for storing and retrieving embeddings.  
- **langchain-tavily==0.2.11** → Integration for the **Tavily API** (specialized web search + retrieval for RAG).  

In [129]:
#Install required packages
!pip install -q openai==1.99.9 \
                langchain==0.3.27 \
                langchain-openai==0.3.30 \
                langchain-community==0.3.27 \
                langgraph==0.6.4 \
                langchain-chroma==0.2.5 \
                chromadb==1.0.16 \
                langchain-tavily==0.2.11

# Setting up the Environment (5 Marks)

## **Importing the relevant packages**

In [130]:
import os
import json
import re
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Azure OpenAI Imports
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# LangChain / LangGraph Imports
from langchain_community.vectorstores import Chroma
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
import chromadb

## **Instantiating the required variables (5 Marks)**

Set the environment variables.

To securely connect with external APIs (like Azure OpenAI and Tavily), we’ll set up our **environment variables**.  
This ensures our API keys are not hard-coded directly in the notebook, keeping them safe.

- **AZURE_OPENAI_API_KEY**
- **AZURE_OPENAI_ENDPOINT**
- **AZURE_API_VERSION**
- **TAVILY_API_KEY**

After running this cell, your notebook will be able to access both **OpenAI** and **Tavily APIs** securely.


In [131]:
import os
import json
# UPDATED IMPORT: Use langchain_chroma instead of langchain_community to fix the warning
from langchain_chroma import Chroma 
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
import chromadb

# --- 1. Load Configurations from JSON files ---
with open('config_gpt.json', 'r') as f:
    config_gpt = json.load(f)

with open('config.json', 'r') as f:
    config_emb = json.load(f)

# --- 2. Set Environment Variables ---

# Azure Keys (from config_gpt.json)
os.environ["AZURE_OPENAI_API_KEY"] = config_gpt["AZURE_OPENAI_KEY"]
os.environ["AZURE_OPENAI_ENDPOINT"] = config_gpt["AZURE_OPENAI_ENDPOINT"]
os.environ["AZURE_OPENAI_API_VERSION"] = config_gpt["AZURE_OPENAI_APIVERSION"]

# Tavily Key Configuration
# Option A: Load from config_gpt.json (if available there)
os.environ["TAVILY_API_KEY"] = config_gpt.get("TAVILY_API_KEY") 

# Option B: Hardcode it here (Replace with your actual key)
#os.environ["TAVILY_API_KEY"] = "tvly-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"


# --- 3. Instantiate Models ---

# Instantiate the Azure OpenAI LLM
llm = AzureChatOpenAI(
    azure_deployment=config_gpt["CHATGPT_MODEL"], 
    api_version=config_gpt["AZURE_OPENAI_APIVERSION"],
    temperature=0
)

# Instantiate the Azure Embedding Model
embedding_model = AzureOpenAIEmbeddings(
    azure_deployment=config_emb.get("EMBEDDING_MODEL", "text-embedding-3-small"), 
    openai_api_version=config_emb.get("AZURE_OPENAI_APIVERSION", "2024-12-01-preview"),
    azure_endpoint=config_emb.get("AZURE_OPENAI_ENDPOINT", config_gpt["AZURE_OPENAI_ENDPOINT"]),
    api_key=config_emb.get("AZURE_OPENAI_KEY", config_gpt["AZURE_OPENAI_KEY"])
)

# --- 4. Initialize ChromaDB ---
PERSIST_DIRECTORY = "./chroma_db"
collection_name = "competitor_analysis"

# Create a persistent client
chroma_client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)

# Initialize the Vector Store
# Note: The updated Chroma class handles the client connection differently.
# If you encounter issues with the persistent client, you can use the simpler init below:
# vectorstore = Chroma(collection_name=collection_name, embedding_function=embedding_model, persist_directory=PERSIST_DIRECTORY)

vectorstore = Chroma(
    client=chroma_client,
    collection_name=collection_name,
    embedding_function=embedding_model,
)

print("Environment variables set and models instantiated.")

Environment variables set and models instantiated.


# **State Definitions (6 Marks)**

Lets define the Pydantic models for structured outputs - 'QuestionSuggestion' and the overall state - 'CompetitiveAnalysisState'  .
State Definition ensure type safety and structure the data passed between agents and nodes.

## **QuestionSuggestion (3 Marks)**


This model will help ensure that the responses from the LLM follow a **consistent format**, making them easier to validate and use later.

- Create a class that inherits from `BaseModel`.  
- Fields include:
  - **sector** → The industry sector of the company.  
  - **is_valid_company** →  A boolean flag for whether the company name is recognized.  
  - **questions** → A list of suggested competitive analysis questions.  
  - **error_message** → Optional field to capture any errors that occur.  

- The `Field()` argument ensures that **default values** and **descriptions** are available for each attribute.  


In [132]:
class QuestionSuggestion(BaseModel):
    """
    Structured output model for competitive analysis questions
    """
    sector: str = Field(
        description="The industry sector of the company."
    )
    is_valid_company: bool = Field(
        description="A boolean flag for whether the company name is recognized."
    )
    questions: List[str] = Field(
        description="A list of suggested competitive analysis questions."
    )
    error_message: Optional[str] = Field(
        default=None, 
        description="Optional field to capture any errors that occur."
    )

## **Competitive Analysis State (3 marks)**

Next, we’ll create a state object for keeping track off the competitive analysis that keeps track of all the information our pipeline generates and updates during the competitive analysis workflow.

This will act as a “shared memory” object, making it easier to pass data between steps in the system.

- Fields include:
  - **company_name** → The company being analyzed.  
  - **max_num_of_questions** → The maximum number of questions to generate (use a constant as the default).  
  - **sector** → The industry sector (optional).  
  - **is_valid_company** → Boolean flag to mark if the company is recognized (optional).  
  - **question_list** → List of generated analysis questions (optional).  
  - **qna_results** → Stores answers to questions as a list of dictionaries (optional).  
  - **vectorstore** → Chroma vector database instance for retrieval.  
  - **chromadb_insert_status** → Tracks whether data was successfully stored in Chroma (optional).  
  - **report** → Final competitive analysis report (optional).  
  - **error_message** → Captures any error details (optional).

- The `model_config` setting allows **non-Pydantic types** (e.g., Chroma objects).


In [133]:
class CompetitiveAnalysisState(BaseModel):
    """
    Shared memory state for the competitive analysis workflow.
    Tracks all information generated and updated during the pipeline.
    """
    # Required Fields
    company_name: str = Field(
        description="The company being analyzed."
    )
    max_num_of_questions: int = Field(
        default=5, 
        description="The maximum number of questions to generate."
    )
    
    # Optional Fields
    sector: Optional[str] = Field(
        default=None, 
        description="The industry sector (optional)."
    )
    is_valid_company: Optional[bool] = Field(
        default=None, 
        description="Boolean flag to mark if the company is recognized (optional)."
    )
    question_list: Optional[List[str]] = Field(
        default=None, 
        description="List of generated analysis questions (optional)."
    )
    qna_results: Optional[List[Dict[str, str]]] = Field(
        default=None, 
        description="Stores answers to questions as a list of dictionaries (optional)."
    )
    
    # Complex Types (Non-Pydantic)
    vectorstore: Any = Field(
        description="Chroma vector database instance for retrieval."
    )
    
    # Status and Outputs
    chromadb_insert_status: Optional[bool] = Field(
        default=None, 
        description="Tracks whether data was successfully stored in Chroma (optional)."
    )
    report: Optional[str] = Field(
        default=None, 
        description="Final competitive analysis report (optional)."
    )
    error_message: Optional[str] = Field(
        default=None, 
        description="Captures any error details (optional)."
    )

    # Configuration to allow arbitrary types like the Chroma vectorstore object
    model_config = {
        "arbitrary_types_allowed": True
    }

# **Tools (13 Marks)**

## **Tavily Search Tool (1 Mark)**

To enrich our competitive analysis, **Tavily Search** will be used to fetch **relevant company information** that can later be embedded into our vector store and used for RAG.
The following code
- Initializes a `TavilySearch` instance with the following parameters:
  - **max_results** → The maximum number of results to return  
  - **include_answer** → Whether to include a synthesized answer from Tavily.  
  - **include_raw_content** → Whether to return raw page content along with the results.
  - **search_depth** → Set the level of search (`"basic"` or `"advanced"`).  

In [134]:
# Updated Import: Use 'TavilySearch' from the new package
from langchain_tavily import TavilySearch

# Initialize the Tavily search tool wrapper
# Note: The class name is now TavilySearch
tavily_tool = TavilySearch(
    max_results=3,
    include_answer=True,
    include_raw_content=True,
    search_depth="advanced"
)

Below section defines the tools used by the agents for,
- Suggesting questions,
- Fetching search results,
- Storing data in ChromaDB, and
- Generating the report.

## **Suggest Questions Tool (3 marks)**


The Suggest Questions Tool is a **custom LangChain tool** that uses an LLM to generate structured competitive analysis questions for a given company.  

This tool will serve as the **entry point** in our pipeline for validating company information and generating insightful prompts for deeper analysis.

The function `suggest_questions` contains a  **prompt** that instructs the LLM to:
     - Validate if the company is real/known.  
     - Identify its sector.  
     - Generate targeted competitor analysis questions (e.g., competitors, pricing, supply chain, strengths/weaknesses, market opportunities) and **invokes the LLM** with a structured output format (using the `QuestionSuggestion` model).  
The `@tool` decorator ensures that it can be called by agents within LangChain.


In [135]:
@tool
def suggest_questions(company_name: str, max_questions: int = 5) -> str:
    """
    Generates structured competitive analysis questions for a given company.
    Validates company, identifies sector, and creates questions.
    Returns a JSON string of the QuestionSuggestion model.
    """
    prompt = f"""
    You are an expert market analyst. 
    1. Validate if '{company_name}' is a real, known company.
    2. Identify its industry sector.
    3. Generate {max_questions} specific strategic questions to analyze its competitors (e.g., market share, pricing, unique selling points, supply chain, strengths/weaknesses).
    
    Return ONLY a JSON object with keys: 'sector', 'is_valid_company', 'questions', 'error_message'.
    """
    # Invoke LLM with structured output requirement using the Pydantic model defined earlier
    structured_llm = llm.with_structured_output(QuestionSuggestion)
    result = structured_llm.invoke(prompt)
    
    # Return as JSON string to be parsed by the node
    return result.model_dump_json()

## **Fetch Search Result Tool (3 Marks)**

The Fetch Search Result tool will take each question, query the **Tavily Search API**, and collect the results.

The function **iterates** through the `question_list` and for each question:
     - Call `search.invoke({"query": question})` to get Tavily search results.  
     - Check if the result contains an `"answer"`.  
     - Store the pair (`question`, `answer`) in a list of dictionaries.  
Additionally, the function can:
- **Handle missing answers** by storing an empty string.
- Handle errors** with a `try/except` block and log them for debugging.  
- Return the **final results as a JSON string**.

In [136]:
@tool
def fetch_search_results(questions: List[str]) -> str:
    """
    Iterates through questions, queries Tavily, and collects answers.
    Returns a JSON string of a list of dictionaries.
    """
    results = []
    print(f"Fetching results for {len(questions)} questions...")
    
    for q in questions:
        try:
            print(f"Querying: {q}...")
            # Invoke the Tavily tool
            search_response = tavily_tool.invoke({"query": q})
            
            answer_text = ""
            
            # --- ROBUST EXTRACTION LOGIC ---
            
            # Case 1: Response is a plain String
            if isinstance(search_response, str):
                answer_text = search_response
                
            # Case 2: Response is a List
            elif isinstance(search_response, list) and len(search_response) > 0:
                first_result = search_response[0]
                
                # Sub-case 2A: List of Dictionaries (Standard JSON)
                if isinstance(first_result, dict):
                    answer_text = first_result.get('content') or first_result.get('body') or first_result.get('answer') or ""
                
                # Sub-case 2B: List of Document Objects (LangChain Documents)
                # This is common with the new langchain_tavily package
                elif hasattr(first_result, 'page_content'):
                    answer_text = first_result.page_content
                    
            # Case 3: Response is a single Dictionary
            elif isinstance(search_response, dict):
                answer_text = search_response.get('content') or search_response.get('body') or search_response.get('answer') or ""

            # -------------------------------

            if answer_text:
                print(f"  -> Found answer: {len(answer_text)} chars")
                results.append({"question": q, "answer": answer_text})
            else:
                print(f"  -> Warning: No text extracted. Raw type: {type(search_response)}")
                results.append({"question": q, "answer": ""})
                
        except Exception as e:
            print(f"  -> Error searching '{q}': {e}")
            results.append({"question": q, "answer": "", "error": str(e)})

    return json.dumps(results)

## **Store in ChromaDB Tool (4 Marks)**

After fetching question-answer pairs, they must be stored in **ChromaDB** so they can later be retrieved during the analysis.  
This tool will embed the answers and insert them into the vector store along with metadata for traceability.

The function `store_in_chromadb`:
- Loop through each question–answer dictionary.  
- **Skip empty answers** (don’t embed/store them).  
- For valid answers:  
    - Create a `documents` list containing the answer text.  
    - Create `metadatas` with fields such as:  
        - `question` → the original question  
        - `source_type` → `"answer"`  
        - `search_success` → whether a valid answer was found  
    - Create unique `ids` for each entry (e.g., `"doc_1"`, `"doc_2"`).  
- Use `vectorstore.add_texts()` to insert documents, metadata, and IDs into ChromaDB.  

In [137]:
@tool
def store_in_chromadb(qna_list_json: str) -> str:
    """
    Stores question-answer pairs in ChromaDB.
    Expects a JSON string of list of dicts.
    """
    try:
        # Parse the input JSON string into a list of dictionaries
        qna_list = json.loads(qna_list_json)
        
        documents = []
        metadatas = []
        ids = []
        
        # Loop through each question-answer pair
        for idx, item in enumerate(qna_list):
            answer = item.get("answer", "")
            question = item.get("question", "")
            
            # Skip empty answers (don't embed/store them)
            if answer and answer.strip():
                # Add the answer text to the documents list
                documents.append(answer)
                
                # Create metadata for traceability
                metadatas.append({
                    "question": question,
                    "source_type": "answer",
                    "search_success": True
                })
                
                # Create a unique ID for each entry (using hash ensures uniqueness per question)
                ids.append(f"doc_{idx}_{hash(question)}")
        
        # Insert into ChromaDB if we have valid documents
        if documents:
            vectorstore.add_texts(texts=documents, metadatas=metadatas, ids=ids)
            return json.dumps({"status": "success", "inserted_count": len(documents)})
        else:
            return json.dumps({"status": "warning", "message": "No valid documents to insert"})
            
    except Exception as e:
        # Handle and return any errors
        return json.dumps({"status": "error", "message": str(e)})

## **Generate Report Tool (2 Marks)**

The `generate_report` tool will be used to define precisely how to use retrieved context for generating the report by
- using the retrieved context as the primary source and comparing competitors across dimensions (market share, products, pricing, tech, supply chain, customer sentiment).
- highlighting differentiators and actionable strategies.
- structuring the final output into the named sections and citing context where relevant.

The functions builds a `human` message that requests a report for the given `company_name` and `sector` and includes a `{context}` placeholder which will be filled at runtime with retrieved documents from the vectorstore and combine messages into a `ChatPromptTemplate.from_messages([...])`

In [138]:
@tool
def generate_report(company_name: str, sector: str) -> ChatPromptTemplate:
    """
    Defines the prompt structure for the final report.
    Returns a ChatPromptTemplate with a {context} placeholder.
    """
    
    # 1. Define the Human Message with instructions
    human_template = f"""
    You are a Strategy Consultant. Write a detailed Competitor Analysis Report for {company_name} in the {sector} sector.
    
    Use the following retrieved context as your primary source:
    {{context}}
    
    Compare {company_name} against its competitors across these dimensions:
    - Market Share & Positioning
    - Key Products & Pricing
    - Technology & Innovation
    - Supply Chain & Operations
    - Customer Sentiment
    
    Highlight key differentiators and provide actionable strategies.
    
    Structure the final output into these named sections:
    1. Executive Summary
    2. Company Overview & Sector
    3. Key Competitors & Market Position
    4. Strengths & Weaknesses (SWOT Analysis)
    5. Market Strategies & Differentiators
    6. Strategic Recommendations
    
    Format the output in Markdown. Cite the context where relevant.
    """
    
    # 2. Combine into a ChatPromptTemplate
    # The system uses this object to fill in the {context} variable at runtime
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional market research analyst."),
        ("human", human_template)
    ])
    
    return prompt

# **Agents (26 Marks)**

## **Initialize the Agents (2 Marks)**

Below section creates reactive agents using LangChain's create_react_agent. Each agent is assigned specific tools:
- question_generator_agent: Uses suggest_questions tool.
- data_retrieval_storage_agent: Uses fetch_search_results and store_in_chromadb tool.
- report_drafter_agent: Uses generate_report tool.

### Questions Generator Agent

In [139]:
# Initialize the Question Generator Agent with the suggest_questions tool
question_generator_agent = create_react_agent(llm, tools=[suggest_questions])

### Data Retrieval & Storage Agent

In [140]:
# Initialize the Data Retrieval Agent with search and storage tools
data_retrieval_storage_agent = create_react_agent(llm, tools=[fetch_search_results, store_in_chromadb])

### Report Drafter Agent

In [141]:
# Initialize the Report Drafter Agent with the generate_report tool
report_drafter_agent = create_react_agent(llm, tools=[generate_report])

# **Defining the Nodes (18 Marks)**

## **Question Generator Node (6 Marks)**

Question Generator Node - Generates questions and updates state.

The run_question_generator node invokes the question-generation agent to produce structured JSON output and then parses that output to update the pipeline state by:
- invoking the agent with a short system+human message (company name, max questions) and retrieving the agent response.
- searching responses in reverse for the last JSON block (to find the structured payload) and using json.loads to parse it.
- updating CompetitiveAnalysisState fields: sector, is_valid_company, question_list, and error_message.
- handling errors and parsing failures gracefully by logging the issue and setting safe defaults (sector="", is_valid_company=False, question_list=[], error_message=<error>).
- returning the updated state for downstream nodes to consume.

In [142]:
def run_question_generator(state: CompetitiveAnalysisState) -> CompetitiveAnalysisState:
    print("--- Node: Question Generator ---")
    
    # 1. Invoke the agent with a short system+human message
    msg = f"Generate competitor analysis questions for company: {state.company_name}. Max questions: {state.max_num_of_questions}"
    result = question_generator_agent.invoke({"messages": [HumanMessage(content=msg)]})
    
    # Helper function to extract JSON from text (handles code blocks)
    def extract_json(text):
        try:
            # Look for JSON code blocks first
            matches = list(re.finditer(r"```json(.*?)```", text, re.DOTALL))
            if matches:
                return json.loads(matches[-1].group(1).strip())
            # Fallback: Look for raw JSON object
            match = re.search(r"(\{.*\})", text, re.DOTALL)
            if match:
                return json.loads(match.group(1))
            # Fallback: Try parsing the whole text
            return json.loads(text)
        except:
            return None

    parsed_data = None

    # 2. Search responses in reverse for the last JSON block
    # We iterate backwards to find the final structured output from the LLM/Tool
    for m in reversed(result["messages"]):
        if m.content and (parsed_data := extract_json(m.content)):
            break
            
    # 3. Update CompetitiveAnalysisState fields or Handle Errors
    if parsed_data:
        # Successfully parsed JSON, update state
        return state.model_copy(update={
            "sector": parsed_data.get("sector", ""),
            "is_valid_company": parsed_data.get("is_valid_company", False),
            "question_list": parsed_data.get("questions", []),
            "error_message": parsed_data.get("error_message")
        })
    else:
        # 4. Handle errors and parsing failures gracefully
        error_msg = "Failed to parse Question Generator output. No valid JSON found."
        print(f"Error: {error_msg}")
        
        return state.model_copy(update={
            "sector": "",
            "is_valid_company": False,
            "question_list": [],
            "error_message": error_msg
        })

## **Data Retrieval and Storage Node (6 Marks)**

This Node fetches and stores the search results.

The run_data_retrieval_storage node fetches answers for generated questions and stores them in ChromaDB, then updates the pipeline state by:
- verifying questions exist and short-circuiting with an error if none were generated.
- invoking the data_retrieval_storage agent with a short system+human message (questions) and retrieving the agent response.
- parsing the last message as JSON to populate state.qna_results (the list of question–answer dicts).
- extracting the `store_in_chromadb` tool call output from `response["messages"][-1].tool_calls` (if present) to set state.chromadb_insert_status, with a fallback to read `chromadb_insert_status` from the parsed last message.
handling JSON parse errors by setting state.error_message to a descriptive message.
- returning the updated CompetitiveAnalysisState for downstream nodes to consume.


In [143]:
def run_data_retrieval_storage(state: CompetitiveAnalysisState) -> CompetitiveAnalysisState:
    print("--- Node: Data Retrieval & Storage ---")
    
    # 1. Verify questions exist and short-circuit with an error if none were generated
    if not state.question_list:
        error_msg = "No questions to search. Question generation might have failed."
        return state.model_copy(update={"error_message": error_msg})

    # 2. Invoke the agent with a short system+human message (questions)
    questions_str = json.dumps(state.question_list)
    msg = f"Fetch search results for these questions and store them in ChromaDB: {questions_str}"
    
    result = data_retrieval_storage_agent.invoke({"messages": [HumanMessage(content=msg)]})
    
    # 3. Parse the results to populate state
    qna_results = []
    insert_status = False
    error_message = None

    try:
        # We need to extract data from the tool calls in the agent's history
        for m in result['messages']:
            if m.type == 'tool':
                # Check for fetch_search_results tool output
                if m.name == 'fetch_search_results':
                    try:
                        qna_results = json.loads(m.content)
                    except json.JSONDecodeError:
                        print("Warning: Failed to parse fetch_search_results output.")

                # Check for store_in_chromadb tool output
                # 4. Extracting the store_in_chromadb tool call output to set state.chromadb_insert_status
                if m.name == 'store_in_chromadb':
                    try:
                        status_json = json.loads(m.content)
                        if status_json.get("status") == "success":
                            insert_status = True
                    except json.JSONDecodeError:
                        print("Warning: Failed to parse store_in_chromadb output.")
                        
        # Fallback: check the last message if tool outputs weren't captured directly
        # (Sometimes agents summarize the tool output in the final message)
        last_msg = result["messages"][-1]
        if not insert_status and "successfully stored" in last_msg.content.lower():
             insert_status = True

    except Exception as e:
        error_message = f"Error processing data retrieval results: {str(e)}"

    # 5. Return the updated CompetitiveAnalysisState for downstream nodes
    return state.model_copy(update={
        "qna_results": qna_results,
        "chromadb_insert_status": insert_status,
        "vectorstore": vectorstore, # Ensure vectorstore is passed along
        "error_message": error_message or state.error_message # Preserve previous errors if new one is None
    })

## **Report Drafter Node (6 Marks)**

Below Node generates the report leveraging RAG and prints a preview.

The run_report_drafter node builds a RAG-powered report by retrieving stored Q&A context and invoking the report-generation prompt, then updates the pipeline state by:
- creating a retriever from state.vectorstore (search_kwargs={"k": 10}) and querying it for competitive-analysis documents for the company.
- handling the "no documents" case by setting state.report and state.error_message and short-circuiting.
- concatenating retrieved documents' `page_content` into a single `context` string for the LLM.
- invoking the `generate_report` tool to obtain a `ChatPromptTemplate` (expects a prompt template, not a final string).
- validating the returned object is a `ChatPromptTemplate`; on mismatch, set `state.error_message` and `state.report` appropriately and return.
- composing the prompt template with the LLM (e.g., `chain = prompt_template | llm`) and invoking the chain with `{"context": context}` to produce the report.
storing the generated report in `state.report`, logging a brief preview, and handling any exceptions by setting `state.error_message` and a failure `state.report`.
- returning the updated CompetitiveAnalysisState for downstream consumption.


In [144]:
def run_report_drafter(state: CompetitiveAnalysisState) -> CompetitiveAnalysisState:
    print("--- Node: Report Drafter ---")
    
    # 1. Create a retriever from state.vectorstore
    if not state.vectorstore:
        error_msg = "Vectorstore not initialized. Cannot retrieve data."
        return state.model_copy(update={"error_message": error_msg, "report": "Report generation failed."})
        
    retriever = state.vectorstore.as_retriever(search_kwargs={"k": 10})
    
    # Query it for competitive-analysis documents for the company
    docs = retriever.invoke(f"{state.company_name} competitor analysis")
    
    # 2. Handle the "no documents" case
    if not docs:
         error_msg = "No relevant documents found in RAG context."
         return state.model_copy(update={"error_message": error_msg, "report": "No data available to generate report."})
    
    # 3. Concatenate retrieved documents' page_content into a single context string
    context_text = "\n\n".join([d.page_content for d in docs])
    
    # 4. Invoke the generate_report tool to obtain a ChatPromptTemplate
    try:
        # FIX: Use .invoke() with a dictionary, instead of calling it like a function
        prompt_template = generate_report.invoke({
            "company_name": state.company_name, 
            "sector": state.sector or "Unknown"
        })
        
        # 5. Validate the returned object is a ChatPromptTemplate
        if not isinstance(prompt_template, ChatPromptTemplate):
            # Sometimes tools return the artifact wrapped; if so, adjust logic or ensure tool definition returns object
            raise ValueError(f"generate_report tool returned unexpected type: {type(prompt_template)}")
            
        # 6. Compose the prompt template with the LLM and invoke the chain
        chain = prompt_template | llm
        
        response = chain.invoke({
            "context": context_text
        })
        
        final_report = response.content
        print("Report generated successfully (preview):", final_report[:100], "...")
        
        # 7. Store the generated report and return state
        return state.model_copy(update={"report": final_report})
        
    except Exception as e:
        # Handle exceptions by setting error_message and a failure report
        error_msg = f"Report generation failed: {str(e)}"
        print(error_msg)
        return state.model_copy(update={"error_message": error_msg, "report": "Report generation failed due to error."})

# **Workflow (5 Marks)**

Building the StateGraph workflow using LangGraph.

Implement the "Sequential design pattern" by adding nodes for each step (question generation, data retrieval/storage, report drafting) in sequence and connects them with edges to define the sequential flow from START to END.

In [145]:
# --- Build StateGraph ---
workflow = StateGraph(CompetitiveAnalysisState)

# Add Nodes
workflow.add_node("question_gen", run_question_generator)
workflow.add_node("data_process", run_data_retrieval_storage)
workflow.add_node("report_draft", run_report_drafter)

# Add Edges (Sequential Pattern)
# START -> Question Generation
workflow.set_entry_point("question_gen")

# Question Generation -> Data Retrieval/Storage
workflow.add_edge("question_gen", "data_process")

# Data Retrieval/Storage -> Report Drafting
workflow.add_edge("data_process", "report_draft")

# Report Drafting -> END
workflow.add_edge("report_draft", END)

# Compile the graph into an executable application
app = workflow.compile()
print("Workflow compiled successfully.")

Workflow compiled successfully.


# **Execution (1 Marks)**

Invoking the workflow with the company  name as input.

In [146]:
# --- Execute Workflow ---

# Define initial state with the company to analyze
# Using Tesla as an example here
initial_state = CompetitiveAnalysisState(
    company_name="Tesla", 
    vectorstore=vectorstore # Pass the initialized chroma instance
)

print(f"Starting Multi-Agent RAG Workflow for: {initial_state.company_name}...")

# Run the graph
final_state = app.invoke(initial_state)

print("Workflow execution completed.")

Starting Multi-Agent RAG Workflow for: Tesla...
--- Node: Question Generator ---
--- Node: Data Retrieval & Storage ---
Fetching results for 5 questions...
Querying: What is Tesla's current market share compared to its main competitors like Ford and General Motors?...
  -> Found answer: 181 chars
Querying: How does Tesla's pricing strategy compare to that of its competitors in the electric vehicle market?...
  -> Found answer: 298 chars
Querying: What unique selling points does Tesla offer that differentiate it from other electric vehicle manufacturers?...
  -> Found answer: 200 chars
Querying: How does Tesla's supply chain management impact its production efficiency compared to competitors?...
  -> Found answer: 264 chars
Querying: What are the key strengths and weaknesses of Tesla in relation to its competitors in the automotive industry?...
  -> Warning: No text extracted. Raw type: <class 'dict'>
--- Node: Report Drafter ---
Report generated successfully (preview): # Competitor Ana

For better readability, displaying the report in Markdown format

In [147]:
from IPython.display import Markdown, display

# Check if a report was generated and display it
if final_state.get("report"):
    display(Markdown(final_state["report"]))
else:
    print("No report generated.")
    # Check for errors if report is missing
    if final_state.get("error_message"):
        print(f"Error encountered: {final_state['error_message']}")

# Competitor Analysis Report for Tesla in the Automotive and Energy Sector

## 1. Executive Summary
Tesla, a leader in the electric vehicle (EV) and energy sectors, holds a significant market share and is recognized for its innovative technology and strong brand presence. However, it faces increasing competition as traditional automakers ramp up their EV offerings. This report analyzes Tesla's position against key competitors, focusing on market share, product offerings, technology, supply chain operations, and customer sentiment. The findings highlight Tesla's strengths and weaknesses, providing actionable strategies to maintain its competitive edge.

## 2. Company Overview & Sector
Tesla, Inc. is an American electric vehicle and clean energy company founded in 2003. It designs, manufactures, and sells electric vehicles, battery energy storage, and solar products. The automotive sector is undergoing a significant transformation, with a growing emphasis on sustainability and electric mobility. Tesla's innovative approach has positioned it as a frontrunner in this transition, holding 41% of the U.S. EV market share.

## 3. Key Competitors & Market Position
Tesla's primary competitors in the EV market include:

- **General Motors (GM)**: Holds 15% of the U.S. EV market share. GM is investing heavily in electric vehicles, with plans to launch multiple models in the coming years.
- **Ford**: Currently experiencing declining sales, Ford is also pivoting towards electric vehicles with its Mustang Mach-E and F-150 Lightning models.
- **Rivian and Lucid Motors**: Emerging players focusing on high-end electric vehicles, targeting niche markets.

The competition is intensifying as government incentives for EVs wane, prompting traditional automakers to adopt aggressive pricing strategies.

## 4. Strengths & Weaknesses (SWOT Analysis)

### Strengths
- **Strong Brand Recognition**: Tesla is synonymous with electric vehicles, benefiting from a loyal customer base.
- **Innovative Technology**: Advanced autonomous driving capabilities and superior battery performance set Tesla apart.
- **Extensive Charging Network**: Tesla's Supercharger network enhances customer convenience and supports long-distance travel.

### Weaknesses
- **High Prices**: Tesla's vehicles are often priced at a premium, limiting accessibility for some consumers.
- **Declining Vehicle Deliveries**: Recent trends indicate a decrease in vehicle deliveries, which could impact market share.

## 5. Market Strategies & Differentiators
Tesla employs a dynamic pricing strategy, adjusting prices based on market conditions and leveraging government incentives. This contrasts with competitors who often rely on traditional discounting methods. Recent price drops by Tesla aim to regain market share and exert pressure on competitors, potentially accelerating the transition to electric vehicles.

Key differentiators include:
- **Vertical Integration**: Tesla's supply chain management enhances production efficiency, reducing costs and risks compared to competitors reliant on outsourcing.
- **Innovative Product Offerings**: Tesla's focus on advanced technology, such as autonomous driving and superior battery performance, positions it as a leader in innovation.

## 6. Strategic Recommendations
To maintain its competitive edge, Tesla should consider the following strategies:

1. **Expand Product Line**: Introduce more affordable models to capture a broader market segment and counteract declining vehicle deliveries.
2. **Enhance Customer Engagement**: Strengthen customer relationships through improved service offerings and loyalty programs to boost sentiment and retention.
3. **Invest in R&D**: Continue investing in research and development to stay ahead in technology and innovation, particularly in autonomous driving and battery technology.
4. **Leverage Partnerships**: Form strategic partnerships with suppliers and technology firms to enhance supply chain efficiency and reduce costs further.
5. **Focus on Sustainability**: Emphasize sustainability in marketing efforts to align with consumer values and differentiate from traditional automakers.

By implementing these strategies, Tesla can solidify its market position and continue to lead the transition to electric vehicles in the automotive and energy sectors.